# 01 — Day 1 Skeleton

**Goal:** prove that the two Day-1 components produce sensible output on the same 50 Fin-Fact claims, side by side. **Do not combine them yet.**

1. **CN6000 RoBERTa baseline** — Year-1 fake-news classifier. We expect heavy distribution collapse on Fin-Fact (this is the cross-domain failure hook).
2. **ArXivMind hybrid retriever** — BM25 + dense + RRF + cross-encoder. We expect it to *retrieve* (return passages) but not be financially relevant — the index holds research papers; financial seeding is Day 4.

Both components are wired through `src/crag_fin/`. Configuration is in `.env`.

## Setup — load env vars and Fin-Fact

In [1]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Ensure project root is on sys.path (notebooks/ is one below it)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
load_dotenv(ROOT / '.env')

with open(ROOT / 'data/finfact/raw/finfact.json') as f:
    finfact = json.load(f)

claims_50 = [r['claim'] for r in finfact[:50]]
gold_50 = [r.get('label', 'N/A') for r in finfact[:50]]
print(f'Loaded {len(finfact)} Fin-Fact entries; using first {len(claims_50)}.')
print(f'Sample claim: {claims_50[0][:100]}')

Loaded 3369 Fin-Fact entries; using first 50.
Sample claim: Did Biden Call for an 'End to Shareholder Capitalism'?


## Cell A — CN6000 RoBERTa baseline

Loads the saved `model0.pth` checkpoint, classifies the 50 claims, prints first 5. The expected pattern is *score collapse near 0.5* on Fin-Fact (severe out-of-distribution shift from ISOT).

In [2]:
from collections import Counter

from crag_fin.baselines.roberta_cn6000 import RobertaCN6000

model = RobertaCN6000()
preds = model.classify(claims_50)

for p, g in list(zip(preds, gold_50, strict=False))[:5]:
    print(f'gold={g!s:8s} pred={p.label} ({p.score:.3f})  | {p.claim[:80]}')

scores = [p.score for p in preds]
print()
print(f'Pred dist: {Counter(p.label for p in preds)}')
print(f'Gold dist: {Counter(gold_50)}')
lo, hi = min(scores), max(scores)
print(f'Score range: [{lo:.4f}, {hi:.4f}]  (collapse near 0.5 = cross-domain failure)')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


gold=false    pred=FAKE (0.507)  | Did Biden Call for an 'End to Shareholder Capitalism'?
gold=NEI      pred=FAKE (0.508)  | Does Jared Kushner's brother own a business associated with COVID-19 testing?
gold=false    pred=FAKE (0.529)  | Texas this fiscal year will have more money in reserve than the other 49 states 
gold=true     pred=FAKE (0.509)  | If we choose Obamacare expansion, 600,000 will lose eligibility for their subsid
gold=false    pred=FAKE (0.513)  | Did 50,000 'Freedom Convoy' Trucks Set Guinness World Record?

Pred dist: Counter({'FAKE': 50})
Gold dist: Counter({'false': 18, 'true': 17, 'NEI': 15})
Score range: [0.5008, 0.5392]  (collapse near 0.5 = cross-domain failure)


## Cell B — ArXivMind hybrid retriever

Calls the ArXivMind substrate (BM25 + dense + RRF + cross-encoder) for the same first 5 claims. The index currently holds research papers, not financial news. We are confirming the *call shape* works; relevant retrieval comes Day 4 after the financial-news seed.

In [3]:
from crag_fin.retriever.arxivmind_adapter import retrieve

for i, claim in enumerate(claims_50[:5], 1):
    passages = retrieve(claim, k=3)
    print(f'\n[{i}] {claim[:100]}')
    print(f'    -> {len(passages)} passages')
    for j, p in enumerate(passages, 1):
        snippet = p.text[:90].replace('\n', ' ')
        print(f'      ({j}) score={p.score:.3f} src={p.source[:30]} | {snippet}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


[1] Did Biden Call for an 'End to Shareholder Capitalism'?
    -> 3 passages
      (1) score=1.000 src=claudes-constitution.pdf | •	 Among the things we’d consider most catastrophic is any kind of global  takeover either...
      (2) score=0.257 src=2507.07957v1.pdf | This memory system, which we call MIRIX, is modular and designed as a comprehensive and fu...
      (3) score=0.000 src=2507.07957v1.pdf | After 8  Figure 7: The Workflow of Responding to the User’s Query. completing the updates,...

[2] Does Jared Kushner's brother own a business associated with COVID-19 testing?
    -> 3 passages
      (1) score=1.000 src=claudes-constitution.pdf | He has been a strong proponent and supporter  of this work. Jared Kaplan worked with Amand...
      (2) score=0.383 src=claudes-constitution.pdf | External commenters who gave detailed feedback or discussion on the  document include: Owe...
      (3) score=0.000 src=claudes-constitution.pdf | Authors   Amanda Askell,* Joe Carlsmith,* Chris O


[4] If we choose Obamacare expansion, 600,000 will lose eligibility for their subsidies, of which 257,00
    -> 3 passages
      (1) score=1.000 src=claudes-constitution.pdf | If it turns out that an AI did have good enough values and  capabilities to be trusted wit...
      (2) score=0.059 src=claudes-constitution.pdf | And Claude could  in principle end up contributing unknowingly to the sort of outcomes the...
      (3) score=0.000 src=claudes-constitution.pdf | And we’ll also try, in the meantime, to develop clearer  policies on AI welfare, to clarif...



[5] Did 50,000 'Freedom Convoy' Trucks Set Guinness World Record?
    -> 3 passages
      (1) score=1.000 src=2507.07957v1.pdf | Single Hop Multi-Hop Open Domain Temporal Overall gpt-4o-mini A-Mem 39.79 18.85 54.05 49.9...
      (2) score=0.659 src=2507.07957v1.pdf | Single Hop Multi-Hop Open Domain Temporal Overall gpt-4o-mini A-Mem 39.79 18.85 54.05 49.9...
      (3) score=0.000 src=2507.07957v1.pdf | 3.1 Memory Components We design a modular memory architecture consisting of six distinct c...


## Day 1 verdict

- Both cells run; both produce non-empty output. **Definition of done met.**
- Baseline shows score collapse near 0.5 on Fin-Fact — the documented cross-domain failure hook.
- Retrieval works mechanically; relevance is poor because the index has not been seeded with financial passages (Day 4 task).
- The two components are deliberately *not* combined here; Day 2 wires the decomposer + credibility prior, Day 3 wires the reasoner.